# Study 871 — The Rank Effect 🏅

**Do the best- and worst-ranked names in a portfolio go on to *under-earn* the middle?**

Hartzmark (2015) finds that investors disproportionately **sell the best- and
worst-ranked positions** in their portfolio — the salience of the extremes drives the
trade, not the raw return. That should put predictable selling pressure on the top- and
bottom-ranked names. We take the self-contained cross-sectional proxy on a liquid US
cross-section (2010-01-04 → 2026-06-30, 50 names): each day rank by trailing
return, **long the middle, short both tails**, and — crucially — **control for the raw
return level**.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

Look at your portfolio sorted by return. The **top** name and the **bottom** name jump out — they are *salient*. Hartzmark shows investors sell those extremes far more than the boring middle-ranked names, regardless of the actual return level. If everyone dumps the extremes, the top- and bottom-ranked names should face selling pressure and **under-earn the middle** next period. So: rank the cross-section, buy the middle, sell both tails — and make sure the raw return level isn't doing the work.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-1.65, t_nw=-1.86, mid_bps=6.71, ext_bps=8.36, lc_spread_bps=0.11, lc_t_nw=0.18, gross_sharpe=-0.46)
print('long-middle / short-extremes spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  middle book %+.2f bps vs extremes book %+.2f bps'
      % (R['mid_bps'], R['ext_bps']))
print('  AFTER controlling for the raw return level: %+.2f bps/day (t = %+.2f)'
      % (R['lc_spread_bps'], R['lc_t_nw']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long-middle / short-extremes spread: -1.65 bps/day (NW t = -1.86)
  middle book +6.71 bps vs extremes book +8.36 bps
  AFTER controlling for the raw return level: +0.11 bps/day (t = +0.18)
  gross spread Sharpe (before cost): -0.46


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: extreme-ranked names carry a forward penalty) and check the detector recovers it — raw *and* after controlling for the level — and that it stays *silent* on the null (`edge=0`, names still ranked but rank carries no forward information). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from rank_effect import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=871, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=871, n_assets=40, n_days=1500))
print('null world   : raw spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: raw spread NW t = %+.2f  (should light up)' % planted['t_nw'])
print('planted world: level-controlled NW t = %+.2f  (survives the level control)' % planted['lc_t_nw'])

null world   : raw spread NW t = -0.63  (should be ~0)
planted world: raw spread NW t = +4.14  (should light up)
planted world: level-controlled NW t = +2.33  (survives the level control)


## 3. The honest verdict — the famous effect leaves *no* footprint here

On this liquid mega-cap tape the long-middle / short-extremes spread is **-1.65 bps/day** with NW *t* = **-1.86** — the **wrong sign** (the claim wants the extremes to under-earn, i.e. a *positive* spread) and **not significant** (|t| < 2). Worse for the story: once you **control for the raw return level** — the whole point of the rank effect — the spread collapses to **+0.11 bps/day** (*t* = +0.18), a flat zero. The tiny raw tilt was just momentum in the tails, not a rank-position effect. The seeded synthetic control recovers a *planted* rank-extremity relation cleanly, so this is a genuine absence, not a broken sort — the rank effect is a **retail-position, trading-behaviour** phenomenon that leaves no tradable cross-sectional signal on 50 mega-caps. **Signal: None**, **Tradability: Mirage**.